# Investment Committee (aus FinRobots Experiment `portfolio_optimization.py`)

Ein CIO leitet drei Analystengruppen (Market Sentiment, Risk Assessment, Fundamental) mit je einem Senior Analyst. Ergebnis: Bewertung, 6-Monats-Kursziel, Empfehlung. Anpassungen gegenueber dem Original: siehe `lib/investment_committee.py`.

**Kosten:** viele verschachtelte Chats; mit Haiku grob 0,5-2 USD pro Lauf. Neue Firmen kosten 3 sec-api-Calls (10-K-Abschnitte), danach Cache.

In [ ]:
# 1) Pfade und Imports
import os, sys, json, glob
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import autogen
from finrobot.utils import register_keys_from_json
from lib.investment_committee import run_committee, committee_task


In [ ]:
# 2) Modell und Schluessel
MODEL = "anthropic/claude-haiku-5.5"   # z. B. "google/gemini-4-neon"

base = json.load(open(FR / "OAI_CONFIG_LIST", encoding="utf-8"))
entry = next((e for e in base if e["model"] == MODEL), None) or dict(base[0], model=MODEL)  # gleicher OpenRouter-Key
llm_config = {"config_list": [entry], "timeout": 300, "temperature": 0.3,
              "parallel_tool_calls": False, "max_tokens": 8000}
if MODEL.startswith("google/"):
    llm_config["extra_body"] = {"reasoning": {"effort": "low"}}
register_keys_from_json(str(FR / "config_api_keys"))


In [ ]:
# 3) Firma und Geschaeftsjahr des 10-K
COMPANY = "RKLB"
FYEAR = "2025"
out_dir = AIF / "reports" / "committee"
print(committee_task(COMPANY, FYEAR))


In [ ]:
# 4) Komitee starten (max_turns begrenzt die Runden des CIO)
final = run_committee(llm_config, COMPANY, FYEAR, out_dir, max_auto_reply=20, max_turns=30)


In [ ]:
# 5) Ergebnis anzeigen
from IPython.display import Markdown, display
display(Markdown(final))
